In [1]:
from pymongo import MongoClient
import pandas as pd

try: 
    client = MongoClient("localhost", 27017)
    print("Connected successfully!!!") 
except:
    print("Could not connect to MongoDB")

db = client["flask_db"]
activity = db.activity


Connected successfully!!!


/var/folders/b4/tsyrjt4j47766fqchqsyvycm0000gn/T/ipykernel_3344/108219557.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
#project_id = "6516fce1459696e2363e98ef"
project_id = "656fadd102ae94a7686aae62"
#project_id = "640e22cae918523bcee8ca5e"
#project_id = "656a440644dec9f71f2dee44"

query = {"project": project_id, "editingLines": {"$exists": True, "$ne": None}}

cursor = activity.find(query)

df = pd.DataFrame(list(cursor))
#df = df.astype({"text": str, "state": str, "line": str, "username": str, "project": str, "file": str, "message": str})

col_names = df.columns.tolist()
dtypes = df.dtypes
df.head()

,_id,timestamp,text,revision,state,line,username,project,file,editingLines,message,changes,clipboard,cb
0,65708e17ea0333c46f1ce0d5,1701875222539,,"[[-1, % This must be in the first 5 lines to t...",Typing,1,das00015@umn.edu,656fadd102ae94a7686aae62,introduction.tex,[1],Typing,"[(1,0), % This must be in the first 5 lines to...",NaN,NaN
1,65710cb4ea0333c46f1ce11f,1701907635170,%I change the title to be more NLP-ish - Shirl...,"[[0, %I change the title to be more NLP-ish - ...",Typing,139,karin,656fadd102ae94a7686aae62,acl_latex.tex,"[103, 104, 105, 106, 107, 108, 109, 110, 111, ...",Typing,"[(139,0), \n---added]",NaN,NaN
2,65710cbfea0333c46f1ce120,1701907646740,%I change the title to be more NLP-ish - Shirl...,"[[0, %I change the title to be more NLP-ish - ...",Typing,139,karin,656fadd102ae94a7686aae62,acl_latex.tex,"[103, 104, 105, 106, 107, 108, 109, 110, 111, ...",Typing,"[(139,0), \karin{DebL }---added]",NaN,NaN
3,65710cc0ea0333c46f1ce121,1701907647197,%I change the title to be more NLP-ish - Shirl...,"[[0, %I change the title to be more NLP-ish - ...",Typing,139,karin,656fadd102ae94a7686aae62,acl_latex.tex,"[103, 104, 105, 106, 107, 108, 109, 110, 111, ...",Typing,"[(139,0), \karin{DebL }->\karin{DebL}]",NaN,NaN
4,65710cc0ea0333c46f1ce122,1701907648145,%I change the title to be more NLP-ish - Shirl...,"[[0, %I change the title to be more NLP-ish - ...",Typing,139,karin,656fadd102ae94a7686aae62,acl_latex.tex,"[103, 104, 105, 106, 107, 108, 109, 110, 111, ...",Typing,"[(139,0), \karin{DebL}->\karin{Deb: ]",NaN,NaN


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


# Sentences we want sentence embeddings for
sentences = ['This is an example sentence', 'Each sentence is converted']

# Load model from HuggingFace Hub
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')

# Tokenize sentences
encoded_input = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

# Compute token embeddings
with torch.no_grad():
    model_output = model(**encoded_input)

# Perform pooling
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

# Normalize embeddings
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

print("Sentence embeddings:")
print(sentence_embeddings)
